# Waiter's Tips Prediction System
Predict tip amount from bill, party, and service features.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np   # linear algebra
import pandas as pd  # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import warnings, os
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

sns.set_theme(style='whitegrid', font_scale=1.1)
SEED = 42

# ── Load data ──────────────────────────────────────────────────────────────────
DATA_PATH = '/kaggle/input/datasets/samarthahv/waiters-tips-dataset-150k-rows/waiters_tips.csv'
if not os.path.exists(DATA_PATH):
    DATA_PATH = 'waiters_tips.csv'

df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# ── EDA ────────────────────────────────────────────────────────────────────────
print(df.describe().round(2))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Tip distribution
axes[0].hist(df['tip'], bins=50, color='steelblue', edgecolor='white')
axes[0].set(title='Tip Distribution', xlabel='Tip ($)')

# Tip vs Total Bill
axes[1].scatter(df['total_bill'], df['tip'], alpha=0.15, s=4, color='coral')
for pct, ls in [(0.10,':'), (0.15,'--'), (0.20,'-')]:
    x = np.linspace(df['total_bill'].min(), df['total_bill'].max(), 100)
    axes[1].plot(x, x*pct, ls=ls, color='black', lw=0.9, label=f'{int(pct*100)}%')
axes[1].set(title='Tip vs Total Bill', xlabel='Total Bill ($)', ylabel='Tip ($)')
axes[1].legend(fontsize=8)

# Mean tip by service rating
df.groupby('service_rating')['tip'].mean().plot(kind='bar', ax=axes[2],
    color='seagreen', edgecolor='white', rot=0)
axes[2].set(title='Avg Tip by Service Rating', xlabel='Rating', ylabel='Avg Tip ($)')

plt.tight_layout()
plt.show()

In [ ]:
# ── Feature Engineering + Split ───────────────────────────────────────────────
df['bill_per_person'] = df['total_bill'] / df['size']
df['log_bill']        = np.log1p(df['total_bill'])
df['is_weekend']      = df['day'].isin(['Sat', 'Sun']).astype(int)
df['is_dinner']       = (df['time'] == 'Dinner').astype(int)
df['long_wait']       = (df['wait_time'] > 30).astype(int)

CAT = ['sex', 'smoker', 'day', 'time', 'payment_method']
NUM = ['total_bill', 'size', 'service_rating', 'wait_time',
       'bill_per_person', 'log_bill', 'is_weekend', 'is_dinner', 'long_wait']

X = df[CAT + NUM]
y = df['tip']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

# ── Preprocessor ─────────────────────────────────────────────────────────────
pre = ColumnTransformer([
    ('num', StandardScaler(),                           NUM),
    ('cat', OneHotEncoder(handle_unknown='ignore'), CAT),
])

In [ ]:
# ── Train & Compare Models ─────────────────────────────────────────────────────
models = {
    'Ridge'            : Ridge(),
    'Random Forest'    : RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, random_state=SEED),
}

results = {}
for name, model in models.items():
    pipe = Pipeline([('pre', pre), ('model', model)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    results[name] = {
        'MAE' : round(mean_absolute_error(y_test, preds), 4),
        'RMSE': round(np.sqrt(mean_squared_error(y_test, preds)), 4),
        'R2'  : round(r2_score(y_test, preds), 4),
    }
    models[name] = pipe   # store fitted pipeline

res_df = pd.DataFrame(results).T.sort_values('R2', ascending=False)
print(res_df)

# ── Best model ────────────────────────────────────────────────────────────────
best_name = res_df['R2'].idxmax()
best_pipe  = models[best_name]
y_pred     = best_pipe.predict(X_test)

print(f'\nBest: {best_name}')

# ── Actual vs Predicted plot ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, y_pred, alpha=0.2, s=5, color='steelblue')
lim = [y_test.min(), y_test.max()]
axes[0].plot(lim, lim, 'r--', lw=1.5)
axes[0].set(title=f'Actual vs Predicted  (R2={res_df.loc[best_name,"R2"]})',
            xlabel='Actual ($)', ylabel='Predicted ($)')

residuals = y_test.values - y_pred
axes[1].hist(residuals, bins=60, color='coral', edgecolor='white')
axes[1].axvline(0, color='black', lw=1, ls='--')
axes[1].set(title='Residual Distribution', xlabel='Residual ($)')

plt.tight_layout()
plt.show()

In [ ]:
# ── Feature Importance + Save ─────────────────────────────────────────────────
model_step = best_pipe.named_steps['model']
if hasattr(model_step, 'feature_importances_'):
    cat_names = best_pipe.named_steps['pre'] \
        .named_transformers_['cat'].get_feature_names_out(CAT).tolist()
    feat_names = NUM + cat_names
    imp = pd.Series(model_step.feature_importances_, index=feat_names) \
            .sort_values(ascending=False).head(12)

    imp.plot(kind='barh', figsize=(9, 5), color='steelblue', edgecolor='white')
    plt.gca().invert_yaxis()
    plt.title('Top 12 Feature Importances')
    plt.tight_layout()
    plt.show()

# ── Save model ────────────────────────────────────────────────────────────────
joblib.dump(best_pipe, '/kaggle/working/waiter_tips_model.pkl')
print('Model saved -> /kaggle/working/waiter_tips_model.pkl')

# ── Quick prediction example ──────────────────────────────────────────────────
sample = pd.DataFrame([{
    'sex': 'Male', 'smoker': 'No', 'day': 'Sat', 'time': 'Dinner',
    'payment_method': 'Credit Card', 'total_bill': 52.50, 'size': 4,
    'service_rating': 5, 'wait_time': 18,
    'bill_per_person': 52.50/4, 'log_bill': np.log1p(52.50),
    'is_weekend': 1, 'is_dinner': 1, 'long_wait': 0,
}])
pred = best_pipe.predict(sample[CAT + NUM])[0]
print(f'\nSample prediction  ->  Bill: $52.50  |  Predicted Tip: ${pred:.2f}  ({pred/52.50*100:.1f}%)')

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np   # linear algebra
import pandas as pd  # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## 2. Load & Inspect Data <a id='2'></a>

In [ ]:
# ── Basic info ─────────────────────────────────────────────────────────────────
print('=== dtypes & non-null counts ===')
df.info()
print()
print('=== Missing values ===')
print(df.isnull().sum())
print()
print('=== Duplicates ===')
print(f'Duplicate rows: {df.duplicated().sum()}')

In [ ]:
# ── Derived column: tip percentage ────────────────────────────────────────────
df['tip_pct'] = (df['tip'] / df['total_bill'] * 100).round(2)
print('Tip percentage stats:')
print(df['tip_pct'].describe().round(2))

## 3. Exploratory Data Analysis (EDA) <a id='3'></a>

In [ ]:
# ── 3.2 Tip vs Total Bill scatter ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sc = axes[0].scatter(
    df['total_bill'], df['tip'],
    c=df['service_rating'], cmap='RdYlGn', alpha=0.3, s=5, edgecolors='none'
)
plt.colorbar(sc, ax=axes[0], label='Service Rating')
axes[0].set_xlabel('Total Bill ($)')
axes[0].set_ylabel('Tip ($)')
axes[0].set_title('Tip vs Total Bill (coloured by Service Rating)')

# Add 10%, 15%, 20% guide lines
x_line = np.linspace(df['total_bill'].min(), df['total_bill'].max(), 100)
for pct, ls in [(0.10, ':'), (0.15, '--'), (0.20, '-')]:
    axes[0].plot(x_line, x_line * pct, ls=ls, color='black', linewidth=0.8,
                 label=f'{int(pct*100)}%')
axes[0].legend(title='Tip Rate', fontsize=8)

sc2 = axes[1].scatter(
    df['total_bill'], df['tip_pct'],
    c=df['wait_time'], cmap='coolwarm', alpha=0.3, s=5, edgecolors='none'
)
plt.colorbar(sc2, ax=axes[1], label='Wait Time (min)')
axes[1].set_xlabel('Total Bill ($)')
axes[1].set_ylabel('Tip Rate (%)')
axes[1].set_title('Tip Rate vs Total Bill (coloured by Wait Time)')

plt.tight_layout()
plt.show()

In [ ]:
# ── 3.4 Numeric correlations ──────────────────────────────────────────────────
num_cols = ['total_bill', 'size', 'service_rating', 'wait_time', 'tip', 'tip_pct']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
    vmin=-1, vmax=1, linewidths=0.5, ax=ax
)
ax.set_title('Correlation Matrix (Numeric Features)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.6 Wait time vs Tip (service interaction) ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for rating, grp in df.groupby('service_rating'):
    axes[0].scatter(grp['wait_time'], grp['tip'], s=4, alpha=0.25,
                    label=f'Rating {rating}')
axes[0].set_xlabel('Wait Time (min)')
axes[0].set_ylabel('Tip ($)')
axes[0].set_title('Wait Time vs Tip by Service Rating')
axes[0].legend(markerscale=3, fontsize=8)

sns.violinplot(data=df, x='service_rating', y='tip_pct', ax=axes[1],
               palette='RdYlGn', inner='quartile')
axes[1].set_title('Tip Rate Distribution by Service Rating')
axes[1].set_xlabel('Service Rating')
axes[1].set_ylabel('Tip Rate (%)')

plt.tight_layout()
plt.show()

## 4. Feature Engineering <a id='4'></a>

## 5. Preprocessing Pipeline <a id='5'></a>

## 6. Model Training & Comparison <a id='6'></a>

In [ ]:
# ── Visual comparison ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

ordered = results_df.index.tolist()

# R2
bars = axes[0].barh(ordered, results_df.loc[ordered, 'R2'],
                    color=sns.color_palette('viridis', len(ordered)))
axes[0].set_xlabel('R2 Score')
axes[0].set_title('Model Comparison — R2 Score (higher is better)')
axes[0].invert_yaxis()
for bar, val in zip(bars, results_df.loc[ordered, 'R2']):
    axes[0].text(bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
                 f'{val:.4f}', va='center', fontsize=8)

# RMSE
bars2 = axes[1].barh(ordered, results_df.loc[ordered, 'RMSE'],
                     color=sns.color_palette('flare', len(ordered)))
axes[1].set_xlabel('RMSE ($)')
axes[1].set_title('Model Comparison — RMSE (lower is better)')
axes[1].invert_yaxis()
for bar, val in zip(bars2, results_df.loc[ordered, 'RMSE']):
    axes[1].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                 f'{val:.4f}', va='center', fontsize=8)

plt.suptitle('Validation Set Performance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Select best model by R2 on validation set ─────────────────────────────────
best_name = results_df['R2'].idxmax()
print(f'Best model on validation: {best_name}')

# ── Tune the best tree-ensemble model ─────────────────────────────────────────
# (We build a small grid; extend for a full Kaggle run)
if 'XGBoost' in best_name and XGB_AVAILABLE:
    param_grid = {
        'model__n_estimators'  : [300, 500],
        'model__max_depth'     : [4, 6],
        'model__learning_rate' : [0.03, 0.05],
        'model__subsample'     : [0.8],
    }
    tune_model = XGBRegressor(random_state=SEED, eval_metric='rmse', verbosity=0)

elif 'LightGBM' in best_name and LGBM_AVAILABLE:
    param_grid = {
        'model__n_estimators'  : [300, 500],
        'model__num_leaves'    : [63, 127],
        'model__learning_rate' : [0.03, 0.05],
    }
    tune_model = LGBMRegressor(random_state=SEED, n_jobs=-1, verbose=-1)

elif 'Random Forest' in best_name:
    param_grid = {
        'model__n_estimators' : [200, 400],
        'model__max_depth'    : [None, 20],
        'model__min_samples_split': [2, 5],
    }
    tune_model = RandomForestRegressor(random_state=SEED, n_jobs=-1)

elif 'Gradient Boosting' in best_name:
    param_grid = {
        'model__n_estimators'  : [200, 400],
        'model__max_depth'     : [3, 5],
        'model__learning_rate' : [0.05, 0.10],
    }
    tune_model = GradientBoostingRegressor(random_state=SEED)

else:
    # Fallback: tune Random Forest
    param_grid = {
        'model__n_estimators' : [200, 400],
        'model__max_depth'    : [None, 20],
    }
    tune_model = RandomForestRegressor(random_state=SEED, n_jobs=-1)

tune_pipe = Pipeline([
    ('pre',   preprocessor),
    ('model', tune_model)
])

grid_search = GridSearchCV(
    tune_pipe, param_grid,
    cv=3, scoring='r2',
    n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)

print(f'\nBest CV R2 : {grid_search.best_score_:.4f}')
print(f'Best params: {grid_search.best_params_}')

tuned_pipeline = grid_search.best_estimator_

In [ ]:
# ── Evaluate tuned model on held-out TEST set ─────────────────────────────────
y_pred_test = tuned_pipeline.predict(X_test)

mae   = mean_absolute_error(y_test, y_pred_test)
rmse  = np.sqrt(mean_squared_error(y_test, y_pred_test))
r2    = r2_score(y_test, y_pred_test)
mape  = mean_absolute_percentage_error(y_test, y_pred_test) * 100

print('=' * 45)
print(f'  TEST SET RESULTS — {best_name}')
print('=' * 45)
print(f'  MAE  : ${mae:.4f}')
print(f'  RMSE : ${rmse:.4f}')
print(f'  R2   : {r2:.4f}')
print(f'  MAPE : {mape:.2f} %')
print('=' * 45)

In [ ]:
# ── Error buckets ─────────────────────────────────────────────────────────────
abs_errors = np.abs(residuals)
thresholds = [0.25, 0.50, 1.00, 2.00]
print('Prediction accuracy within tolerance:')
for t in thresholds:
    pct = (abs_errors <= t).mean() * 100
    print(f'  Within ${t:.2f} : {pct:.1f}%')

print(f'\nMedian absolute error : ${np.median(abs_errors):.4f}')
print(f'95th pct error        : ${np.percentile(abs_errors, 95):.4f}')

In [ ]:
# ── Extract feature names after preprocessing ─────────────────────────────────
fitted_pre = tuned_pipeline.named_steps['pre']
num_feature_names = NUM_COLS
cat_feature_names = fitted_pre.named_transformers_['cat'].get_feature_names_out(CAT_COLS).tolist()
all_feature_names = num_feature_names + cat_feature_names

# ── Get importances if tree-based model ───────────────────────────────────────
model_step = tuned_pipeline.named_steps['model']

if hasattr(model_step, 'feature_importances_'):
    importances = model_step.feature_importances_
    feat_imp_df = pd.DataFrame({
        'feature'   : all_feature_names,
        'importance': importances
    }).sort_values('importance', ascending=False).head(20)

    fig, ax = plt.subplots(figsize=(10, 7))
    palette = sns.color_palette('viridis', len(feat_imp_df))
    sns.barplot(data=feat_imp_df, x='importance', y='feature',
                palette=palette, ax=ax)
    ax.set_title('Top 20 Feature Importances', fontsize=13, fontweight='bold')
    ax.set_xlabel('Importance Score')
    ax.set_ylabel('')
    plt.tight_layout()
    plt.show()
    print(feat_imp_df.to_string(index=False))
else:
    # For linear models use coefficients
    if hasattr(model_step, 'coef_'):
        coef_df = pd.DataFrame({
            'feature': all_feature_names,
            'coef'   : np.abs(model_step.coef_)
        }).sort_values('coef', ascending=False).head(20)
        print('Top 20 features by |coefficient|:')
        print(coef_df.to_string(index=False))
    else:
        print(f'Model {best_name} does not expose feature importances.')

In [ ]:
def prepare_input(raw: dict) -> pd.DataFrame:
    """Add engineered features to a raw input dict and return a DataFrame."""
    row = raw.copy()
    row['bill_per_person']  = round(row['total_bill'] / row['size'], 2)
    row['is_weekend']       = int(row['day'] in ['Sat', 'Sun'])
    row['is_dinner']        = int(row['time'] == 'Dinner')
    row['high_service']     = int(row['service_rating'] >= 4)
    row['long_wait']        = int(row['wait_time'] > 30)
    row['bill_x_service']   = round(row['total_bill'] * row['service_rating'], 2)
    row['log_total_bill']   = round(np.log1p(row['total_bill']), 4)
    return pd.DataFrame([row])[CAT_COLS + NUM_COLS]


# ── Example predictions ───────────────────────────────────────────────────────
examples = [
    dict(total_bill=52.50, sex='Male',   smoker='No',  day='Sat', time='Dinner',
         size=4, service_rating=5, wait_time=18, payment_method='Credit Card'),
    dict(total_bill=18.90, sex='Female', smoker='No',  day='Thur', time='Lunch',
         size=2, service_rating=3, wait_time=25, payment_method='Cash'),
    dict(total_bill=75.00, sex='Male',   smoker='Yes', day='Sun', time='Dinner',
         size=6, service_rating=4, wait_time=40, payment_method='Credit Card'),
    dict(total_bill=12.00, sex='Female', smoker='No',  day='Fri', time='Lunch',
         size=1, service_rating=2, wait_time=35, payment_method='Debit Card'),
]

print(f'{"Example":<8} {"Bill":>7}  {"Size":>4}  {"Service":>7}  {"Wait":>6}  {"Predicted Tip":>13}  {"Tip %":>6}')
print('-' * 65)
for i, ex in enumerate(examples, 1):
    X_new  = prepare_input(ex)
    pred   = tuned_pipeline.predict(X_new)[0]
    tip_pct = pred / ex['total_bill'] * 100
    print(f'{i:<8} ${ex["total_bill"]:>6.2f}  {ex["size"]:>4}  '
          f'{ex["service_rating"]:>7}  {ex["wait_time"]:>5.0f}m  '
          f'${pred:>11.2f}  {tip_pct:>5.1f}%')

In [ ]:
# ── Save to /kaggle/working/ ───────────────────────────────────────────────────
OUTPUT_DIR = '/kaggle/working'
if not os.path.exists(OUTPUT_DIR):
    OUTPUT_DIR = '.'   # fallback for local runs

model_path = os.path.join(OUTPUT_DIR, 'waiter_tips_model.pkl')
joblib.dump(tuned_pipeline, model_path)
print(f'Model saved -> {model_path}')

# ── Save test predictions ─────────────────────────────────────────────────────
test_preds_df = X_test.copy().reset_index(drop=True)
test_preds_df['actual_tip']    = y_test.values
test_preds_df['predicted_tip'] = np.round(y_pred_test, 2)
test_preds_df['error']         = np.round(test_preds_df['actual_tip'] - test_preds_df['predicted_tip'], 2)

preds_path = os.path.join(OUTPUT_DIR, 'test_predictions.csv')
test_preds_df.to_csv(preds_path, index=False)
print(f'Predictions saved -> {preds_path}')

# ── Final summary card ────────────────────────────────────────────────────────
print()
print('=' * 45)
print('  FINAL SUMMARY')
print('=' * 45)
print(f'  Model   : {best_name}')
print(f'  Dataset : 150,000 rows x 10 columns')
print(f'  Train   : {X_train.shape[0]:,} | Val: {X_val.shape[0]:,} | Test: {X_test.shape[0]:,}')
print(f'  MAE     : ${mae:.4f}')
print(f'  RMSE    : ${rmse:.4f}')
print(f'  R2      : {r2:.4f}')
print(f'  MAPE    : {mape:.2f}%')
print('=' * 45)